In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Conv2D,MaxPool2D,Flatten,Dense,Dropout,concatenate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import BatchNormalization

from PIL import Image
import matplotlib.pyplot as plt



In [ ]:
import zipfile

zip_path = "/content/EuroSAT.zip"
extract_path = "/content/EuroSAT"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:

IMG_SIZE = 64

data = []
labels = []

classes = ["Industrial","Residential","Highway"]

dataset_path = "EuroSAT/2750"

for cls in classes:

    path = os.path.join(dataset_path,cls)

    for img in os.listdir(path)[:700]:

        img_path = os.path.join(path,img)

        image = cv2.imread(img_path)

        if image is None:
            continue

        image = cv2.resize(image,(IMG_SIZE,IMG_SIZE))

        data.append(image)

        if cls in ["Industrial","Highway"]:
            labels.append(1)
        else:
            labels.append(0)

X = np.array(data)/255.0
y = np.array(labels)

print("Dataset:",X.shape)

Dataset: (2100, 64, 64, 3)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
# Data augmentation
datagen = ImageDataGenerator(

    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True

)

datagen.fit(X_train)

In [ ]:

# CNN architecture
image_input = Input(shape=(64,64,3))

x = Conv2D(32,(3,3),activation='relu')(image_input)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Conv2D(64,(3,3),activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Conv2D(128,(3,3),activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPool2D()(x)

x = Flatten()(x)

x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)

feature_layer = Dense(32,activation='relu',name="feature_layer")(x)

output = Dense(1,activation='sigmoid')(feature_layer)

model = Model(inputs=image_input,outputs=output)

In [ ]:

model.compile(
optimizer='adam',
loss='binary_crossentropy',
metrics=['accuracy']
)


In [ ]:

history = model.fit(

    datagen.flow(X_train,y_train,batch_size=32),

    validation_data=(X_test,y_test),

    epochs=10

)

Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 404ms/step - accuracy: 0.9358 - loss: 0.1594 - val_accuracy: 0.7405 - val_loss: 0.8486
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 20s 369ms/step - accuracy: 0.9594 - loss: 0.1407 - val_accuracy: 0.7905 - val_loss: 0.7377
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 401ms/step - accuracy: 0.9579 - loss: 0.1151 - val_accuracy: 0.7690 - val_loss: 1.3866
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 19s 365ms/step - accuracy: 0.9085 - loss: 0.1903 - val_accuracy: 0.8405 - val_loss: 0.7463
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 395ms/step - accuracy: 0.9547 - loss: 0.1612 - val_accuracy: 0.7286 - val_loss: 0.9660
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 20s 369ms/step - accuracy: 0.9522 - loss: 0.1402 - val_accuracy: 0.7595 - val_loss: 1.1419
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 20s 377ms/step - accuracy: 0.9351 - loss: 0.1601 - val_accuracy: 0.9429 - val_loss: 0.1302
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 19s 364ms/step - accuracy: 0.9405 - loss: 0.1431 - val_accu

In [ ]:
loss,accuracy = model.evaluate(X_test,y_test)

print("Accuracy:",accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.8741 - loss: 0.2648
Accuracy: 0.8714285492897034


In [ ]:
import pickle

with open("cnn_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as cnn_model.pkl")

Model saved as cnn_model.pkl


In [ ]:
#feature extracter
%%writefile feature_extractor.py
import numpy as np
import cv2
import pickle
from tensorflow.keras.models import Model

IMG_SIZE = 64

# Load model from pkl
with open("cnn_model.pkl", "rb") as f:
    model = pickle.load(f)

# Create feature extraction model
feature_model = Model(
    inputs=model.input,
    outputs=model.get_layer("feature_layer").output
)

def extract_features(image):

    img = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    img = img/255.0
    img = img.reshape(1,64,64,3)

    features = feature_model.predict(img)[0]

    commercial = np.mean(features[:10])
    traffic = np.mean(features[10:20])
    powerline = np.mean(features[20:32])

    return commercial,traffic,powerline


def predict_location(image):

    img = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    img = img/255.0
    img = img.reshape(1,64,64,3)

    score = model.predict(img)[0][0]

    commercial,traffic,powerline = extract_features(image)

    if score > 0.5:
        label = "Suitable"
    else:
        label = "Not Suitable"

    return label,score,commercial,traffic,powerline

Writing feature_extractor.py


In [ ]:
!pip install streamlit -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 88.9 MB/s eta 0:00:00


In [ ]:
#streamlit code
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from PIL import Image
from feature_extractor import predict_location

st.set_page_config(page_title="EV Charging Hub Predictor")

st.title("⚡ EV Charging Hub Location Predictor")

uploaded_file = st.file_uploader(
"Upload Satellite Image",
type=["jpg","png","jpeg"]
)

if uploaded_file:

    image = Image.open(uploaded_file)

    st.image(image,use_column_width=True)

    image_np = np.array(image)

    if st.button("Analyze Location"):

        label,score,commercial,traffic,powerline = predict_location(image_np)

        st.subheader("Prediction")

        st.write("Suitability Score:",round(score,2))

        st.success(label)

        st.subheader("Extracted Infrastructure Parameters")

        st.write("Commercial Activity:",round(commercial,2))
        st.progress(int(commercial*100))

        st.write("Traffic Density:",round(traffic,2))
        st.progress(int(traffic*100))

        st.write("Powerline Proximity:",round(powerline,2))
        st.progress(int(powerline*100))

Writing app.py


In [ ]:
%%writefile app.py
import streamlit as st

st.title("EV Charging Hub Predictor")

st.write("Streamlit is working successfully!")

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.194.72:8501

  Stopping...
^C
